# Chapter 35
## Periodic Inhibition
- Code by : [Abolfazl Ziaeemehr](https://github.com/Ziaeemehr)


[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ITNG/ModelingNeuralDynamics/blob/main/python/chapter35.ipynb)


## About this chapter

Periodic inhibitory forcing creates recurring response windows rather than
merely lowering firing. Instead of a constant (tonic) inhibitory
conductance, an inhibitory population that fires rhythmically produces a
conductance $g(t)$ that peaks once per cycle and nearly vanishes in
between. A postsynaptic cell then only gets to escape inhibition and fire
during a narrow window each cycle, so its response is timed by the cycle
rather than merely damped by the mean conductance.

The forcing is modeled with a smooth periodic gate,

$$
I_{\rm inh}(t) = g_{\rm inh}\,s(t)\,(E_I - v), \qquad
s(t) \propto \exp\!\big(\alpha\cos^2(\pi t / T)\big) - 1,
$$

normalized so that its time average over one period equals a chosen mean
conductance $\bar g$; larger $\alpha$ sharpens the pulses without changing
that mean. Sweeping applied current and counting spikes per observation
time under this forcing (instead of under the constant mean $\bar g$)
produces an inhibited f-I relation.

The examples below move from the shape of the periodic gate itself,
through single-trace LIF responses (deterministic and noisy) that compare
periodic to tonic (mean) inhibition, to f-I curves that show how periodic
inhibition can make spikes skip cycles rather than simply firing slower --
first for the LIF neuron, then for the conductance-based RTM neuron.

See [`chapter35.md`](chapter35.md) for the full guide, including suggested
order and related chapters.


In [ ]:
import subprocess
import sys
if "google.colab" in sys.modules:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "modelingneuraldynamics"], check=True)

In [ ]:
import math

import numpy as np
from numpy import exp
import matplotlib.pyplot as plt
from ipywidgets import interact
from numba import njit
from numba.extending import register_jitable

## `OSCILLATIONS`: shape of the periodic inhibitory gate

The periodic gate $\phi(t) = \exp(\alpha\sin^2(\pi t)) - 1$, normalized to
a unit mean, is the building block used by every periodic-inhibition
example below. As $\alpha$ grows, the gate goes from a nearly flat (tonic)
profile to a train of narrow, sharp pulses -- same time-averaged strength,
very different instantaneous dynamics.


In [ ]:
def phi_of(alpha, t=None):
    """Normalized periodic inhibitory gate exp(alpha*sin^2(pi*t)) - 1,
    with unit mean over one period."""
    if t is None:
        t = np.arange(-250, 251) / 100.
    phi = np.exp(alpha * np.sin(np.pi * t) ** 2) - 1
    return phi / phi[1:].mean()


def plot_oscillations():
    t = np.arange(-250, 251) / 100.
    fig, axes = plt.subplots(3, 1, figsize=(6, 8))
    for ax, alpha, label in zip(axes, [1e-5, 5, 10],
                                 [r'$\alpha \rightarrow 0$', r'$\alpha=5$', r'$\alpha=10$']):
        ax.plot(t, phi_of(alpha, t), '-k', linewidth=2)
        ax.set_title(label)
        ax.axis([-2.5, 2.5, 0, 7])
    plt.tight_layout()
    return fig

In [ ]:
plot_oscillations();

In [ ]:
def plot_phi(alpha=5.0):
    t = np.arange(-250, 251) / 100.
    fig, ax = plt.subplots(figsize=(6, 3))
    ax.plot(t, phi_of(alpha, t), '-k', linewidth=2)
    ax.axis([-2.5, 2.5, 0, 10])
    ax.set_title(rf'$\alpha={alpha:.2f}$')
    plt.tight_layout()
    return fig


interact(plot_phi, alpha=(0.0, 15.0, 0.5));

## `PERIODIC_INHIBITION` and `PERIODIC_INHIBITION_3`: periodic vs. tonic inhibitory forcing

A single LIF neuron receives either a periodic inhibitory conductance
$g(t)$ (blue) built from `phi_of`, or the same conductance's time-average
$\bar g$ held constant (red, "tonic" inhibition). The two examples only
differ in the sharpness parameter $\alpha$ of the periodic gate (5 vs. 1);
both variants are driven by the same shared `run_periodic_inhibition`.


In [ ]:
def periodic_inhibition_conductance(t, alpha=5.0, Period=25.0, g_bar=0.1, N=2000):
    """Periodic inhibitory conductance g(t), normalized so its average
    over one period equals g_bar."""
    m = np.mean(np.exp(alpha * np.cos(np.pi * np.arange(N) / N) ** 2) - 1)
    return g_bar * (np.exp(alpha * np.cos(np.pi * t / Period) ** 2) - 1) / m


def run_periodic_inhibition(I, use_periodic, alpha=5.0, Period=25.0, g_bar=0.1,
                             tau=10.0, t_final=100.0, dt=0.01):
    """Integrate the LIF neuron under periodic (or tonic mean) inhibitory
    conductance. Returns (t, v) segments split at each reset (so each
    inter-spike segment can be plotted separately), the spike times, and
    the firing frequency."""
    dt05 = dt / 2
    m_steps = round(t_final / dt)
    t = np.arange(m_steps + 1) * dt

    def g(tt):
        return periodic_inhibition_conductance(tt, alpha, Period, g_bar) if use_periodic else g_bar

    v = np.zeros(m_steps + 1)
    k_old = 0
    num_spikes = 0
    segments = []
    spike_times = []
    for k in range(1, m_steps + 1):
        g_old = g((k - 1) * dt)
        g_mid = g((k - 0.5) * dt)
        v_inc = -v[k - 1] / tau + I - g_old * v[k - 1]
        v_tmp = v[k - 1] + dt05 * v_inc
        v_inc = -v_tmp / tau + I - g_mid * v_tmp
        v[k] = v[k - 1] + dt * v_inc
        if v[k] > 1:
            segments.append((t[k_old:k + 1], v[k_old:k + 1].copy()))
            k_old = k
            v[k] = 0
            num_spikes += 1
            spike_times.append(k * dt)
    segments.append((t[k_old:], v[k_old:]))
    freq = num_spikes / t_final * 1000
    return segments, spike_times, freq


def plot_periodic_inhibition(alpha=5.0, I_values=(0.15, 0.2), t_final=100.0):
    dt = 0.01
    g_bar = 0.1
    t = np.arange(round(t_final / dt) + 1) * dt

    fig, axes = plt.subplots(3, 1, figsize=(7, 8))
    g_vals = periodic_inhibition_conductance(t, alpha=alpha, g_bar=g_bar)
    axes[0].plot(t, g_vals, '-b', linewidth=2)
    axes[0].plot([0, t_final], [g_bar, g_bar], '-r', linewidth=2)
    axes[0].set_title(r'$g$ (blue) and $\overline{g}$ (red)')
    axes[0].axis([0, t_final, 0, g_vals.max() * 1.2])

    for ax, I in zip(axes[1:], I_values):
        segs, spikes, freq = run_periodic_inhibition(I, True, alpha=alpha, t_final=t_final)
        for seg_t, seg_v in segs:
            ax.plot(seg_t, seg_v, '-b', linewidth=2)
        for ts in spikes:
            ax.plot([ts, ts], [0, 5], '-b', linewidth=2)
        segs_bar, spikes_bar, freq_bar = run_periodic_inhibition(I, False, alpha=alpha, t_final=t_final)
        for seg_t, seg_v in segs_bar:
            ax.plot(seg_t, seg_v, '-r', linewidth=2)
        for ts in spikes_bar:
            ax.plot([ts, ts], [0, 1], '-r', linewidth=2)
        ax.axis([0, t_final, 0, 6])
        ax.set_title(rf'$v$ (blue) and $\overline{{v}}$ (red), $I={I}$, $\alpha={alpha}$')
        print(f"alpha={alpha}, I={I}: freq={freq:.2f} Hz, freq_bar={freq_bar:.2f} Hz")

    axes[-1].set_xlabel('$t$ [ms]')
    plt.tight_layout()
    return fig

In [ ]:
plot_periodic_inhibition(alpha=5.0);  # PERIODIC_INHIBITION

In [ ]:
plot_periodic_inhibition(alpha=1.0);  # PERIODIC_INHIBITION_3

In [ ]:
interact(lambda alpha=5.0: plot_periodic_inhibition(alpha=alpha), alpha=(0.1, 10.0, 0.1));

## `PERIODIC_INHIBITION_2`: periodic inhibition with a noisy drive

Same LIF neuron and periodic inhibitory conductance as above, but the
applied current is now a constant mean plus an Ornstein-Uhlenbeck noise
process `s_noise` (time constant `tau_noise`, standard deviation
`sigma_noise`). The same noise realization (fixed `seed`) drives both the
periodic and the tonic run, so any difference in output is attributable
to the inhibition, not the noise.


In [ ]:
def run_periodic_inhibition_noisy(use_periodic, alpha=5.0, Period=25.0, g_bar=0.1,
                                   tau=10.0, I=0.1, tau_noise=3.0, sigma_noise=0.08,
                                   t_final=500.0, dt=0.01, seed=63806):
    """Integrate the noisy LIF neuron under periodic (or tonic mean)
    inhibitory conductance; same return convention as
    run_periodic_inhibition. MATLAB's rng('default'); rng(63806) cannot
    be bit-reproduced by NumPy, so this seed is our own and results are
    checked statistically/visually rather than against exact MATLAB
    spike times."""
    rng = np.random.default_rng(seed)
    dt05 = dt / 2
    m_steps = round(t_final / dt)
    gamma = sigma_noise * np.sqrt(1 - np.exp(-2 * dt / tau_noise))
    t = np.arange(m_steps + 1) * dt

    s_noise = np.zeros(m_steps + 1)
    s_noise[0] = sigma_noise * rng.standard_normal()
    for k in range(m_steps):
        s_noise[k + 1] = s_noise[k] * np.exp(-dt / tau_noise) + gamma * rng.standard_normal()

    def g(tt):
        return periodic_inhibition_conductance(tt, alpha, Period, g_bar) if use_periodic else g_bar

    v = np.zeros(m_steps + 1)
    k_old = 0
    num_spikes = 0
    segments = []
    spike_times = []
    for k in range(1, m_steps + 1):
        g_old = g((k - 1) * dt)
        g_mid = g((k - 0.5) * dt)
        v_inc = -v[k - 1] / tau + I + s_noise[k - 1] - g_old * v[k - 1]
        v_tmp = v[k - 1] + dt05 * v_inc
        v_inc = -v_tmp / tau + I + (s_noise[k - 1] + s_noise[k]) / 2 - g_mid * v_tmp
        v[k] = v[k - 1] + dt * v_inc
        if v[k] > 1:
            segments.append((t[k_old:k + 1], v[k_old:k + 1].copy()))
            k_old = k
            v[k] = 0
            num_spikes += 1
            spike_times.append(k * dt)
    segments.append((t[k_old:], v[k_old:]))
    freq = num_spikes / t_final * 1000
    return segments, spike_times, freq


def plot_periodic_inhibition_noisy(alpha=5.0, sigma_noise=0.08, seed=63806):
    t_final = 500.0
    dt = 0.01
    g_bar = 0.1
    m_steps = round(t_final / dt)
    t = np.arange(m_steps + 1) * dt

    fig, axes = plt.subplots(3, 1, figsize=(9, 8))
    axes[0].plot(t, periodic_inhibition_conductance(t, alpha=alpha, g_bar=g_bar), '-b', linewidth=2)
    axes[0].plot(t, g_bar * np.ones(m_steps + 1), '-r', linewidth=2)
    axes[0].set_ylabel('$g$')
    axes[0].axis([0, t_final, 0, 0.5])

    segs, spikes, freq = run_periodic_inhibition_noisy(True, alpha=alpha, sigma_noise=sigma_noise, seed=seed)
    for seg_t, seg_v in segs:
        axes[1].plot(seg_t, seg_v, '-b', linewidth=2)
    for ts in spikes:
        axes[1].plot([ts, ts], [0, 5], '-b', linewidth=2)
    axes[1].axis([0, t_final, -1, 6])
    axes[1].set_ylabel('$v$')
    print(f"freq={freq:.2f} Hz")

    segs_bar, spikes_bar, freq_bar = run_periodic_inhibition_noisy(False, alpha=alpha, sigma_noise=sigma_noise, seed=seed)
    for seg_t, seg_v in segs_bar:
        axes[2].plot(seg_t, seg_v, '-r', linewidth=2)
    for ts in spikes_bar:
        axes[2].plot([ts, ts], [0, 5], '-r', linewidth=2)
    axes[2].axis([0, t_final, -1, 6])
    axes[2].set_ylabel(r'$\overline{v}$')
    axes[2].set_xlabel('$t$ [ms]')
    print(f"freq_bar={freq_bar:.2f} Hz")

    plt.tight_layout()
    return fig

In [ ]:
plot_periodic_inhibition_noisy();

In [ ]:
interact(lambda sigma_noise=0.08: plot_periodic_inhibition_noisy(sigma_noise=sigma_noise),
         sigma_noise=(0.0, 0.2, 0.01));

## `PERIODIC_INHIBITION_F_I_CURVE` and `PERIODIC_INHIBITION_F_I_CURVE_2`: inhibited f-I curves

Sweeping the applied current and counting spikes per observation time
under periodic inhibition gives a step-like f-I curve: firing rate jumps
between plateaus (roughly 40, 80, 120, 160 Hz) corresponding to the
neuron skipping 0, 1, 2, ... inhibitory cycles between spikes, instead of
smoothly tracking the closed-form tonic (mean-conductance) f-I curve
(red). The two examples share `compute_periodic_inhibition_f_i_curve`,
differing only in the gate sharpness $\alpha$ (5 vs. 1). The scan
integrates 100 drive values over 2 s each, so its inner loop is
`@njit`-compiled (verified against a pre-numba Python version at reduced
scale before trusting it).


In [ ]:
# njit without cache=True: this notebook is loaded by exec'ing extracted
# definitions (not imported as a real module), so numba's on-disk cache
# can't re-import it to rebuild the compiled environment; compilation is
# redone per process.
@njit
def _periodic_inhibition_f_i_sweep_jit(i_vec, alpha, period, g_bar, tau, m_steps, dt, dt05,
                                        t_final, norm):
    f_vec = np.zeros(len(i_vec))
    for ij in range(len(i_vec)):
        I = i_vec[ij]
        v = 0.
        num_spikes = 0
        for k in range(1, m_steps + 1):
            g_old = g_bar * (math.exp(alpha * math.pow(math.cos(math.pi * (k - 1) * dt / period), 2.0)) - 1) / norm
            g_mid = g_bar * (math.exp(alpha * math.pow(math.cos(math.pi * (k - 0.5) * dt / period), 2.0)) - 1) / norm
            v_inc = -v / tau + I - g_old * v
            v_tmp = v + dt05 * v_inc
            v_inc = -v_tmp / tau + I - g_mid * v_tmp
            v = v + dt * v_inc
            if v > 1:
                v = 0.
                num_spikes += 1
        f_vec[ij] = num_spikes / t_final * 1000
    return f_vec


def compute_periodic_inhibition_f_i_curve(alpha=5.0, Period=25.0, g_bar=0.1, tau=10.0,
                                            t_final=2000.0, dt=0.01, N=2000, i_values=None):
    """F-I curve of the LIF neuron under periodic inhibition. Returns
    (I_vec, f_vec), one f_vec entry per drive in I_vec (i_values, or a
    default sweep from 0.11 to 0.3)."""
    if i_values is None:
        i_values = np.arange(1, 101) / 100 * 0.2 + 0.1
    norm = np.mean(np.exp(alpha * np.cos(np.pi * np.arange(N) / N) ** 2) - 1)
    m_steps = round(t_final / dt)
    dt05 = dt / 2
    f_vec = _periodic_inhibition_f_i_sweep_jit(i_values, alpha, Period, g_bar, tau, m_steps, dt, dt05,
                                                t_final, norm)
    return i_values, f_vec


def compute_periodic_inhibition_f_i_curve_2(i_values=None):
    return compute_periodic_inhibition_f_i_curve(alpha=1.0, i_values=i_values)


def plot_periodic_inhibition_f_i_curve(I_vec, f_vec, tau=10.0, g_bar=0.1):
    fig, ax = plt.subplots(figsize=(7, 5))

    L = len(I_vec)
    for k in range(L - 1):
        if f_vec[k + 1] - f_vec[k] <= 1:
            ax.plot([I_vec[k], I_vec[k + 1]], [f_vec[k], f_vec[k + 1]], '-b', linewidth=2)
        else:
            mid = (I_vec[k] + I_vec[k + 1]) / 2
            ax.plot([mid, mid], [f_vec[k], f_vec[k + 1]], '--b', linewidth=1)

    # closed-form F-I curve of the LIF neuron under a constant (mean)
    # inhibitory conductance g_bar
    f = np.arange(1, 201)
    A = f / 1000 / (1 + tau * g_bar)
    A = 1 / A
    A = A / tau
    A = np.exp(A)
    I_closed = A * (1 + tau * g_bar) / tau / (A - 1)
    ax.plot(I_closed, f, '-r', linewidth=2)
    ax.axis([I_vec.min(), I_vec.max(), 0, 200])
    ax.set_xlabel('$I$')
    ax.set_ylabel('$f$ [Hz]')

    I_onset_tonic = 1 / tau + g_bar
    ax.plot([I_vec.min(), I_onset_tonic], [0, 0], '-r', linewidth=4)
    ax.plot(I_onset_tonic, 0, '.r', markersize=20)

    for k in range(L - 1):
        if f_vec[k] == 0 and f_vec[k + 1] > 0:
            I_onset = (I_vec[k] + I_vec[k + 1]) / 2
            ax.plot(I_onset, 0, '.b', markersize=20)

    plt.tight_layout()
    return fig

In [ ]:
plot_periodic_inhibition_f_i_curve(*compute_periodic_inhibition_f_i_curve());  # PERIODIC_INHIBITION_F_I_CURVE

In [ ]:
plot_periodic_inhibition_f_i_curve(*compute_periodic_inhibition_f_i_curve_2());  # PERIODIC_INHIBITION_F_I_CURVE_2

In [ ]:
interact(lambda alpha=5.0: plot_periodic_inhibition_f_i_curve(*compute_periodic_inhibition_f_i_curve(alpha=alpha)),
         alpha=(0.0, 10.0, 0.5));

## `RTM_F_I_CURVE_WITH_INHIBITION` and `RTM_F_I_CURVE_WITH_INHIBITION_2`: RTM f-I curves under inhibition

Same comparison (tonic vs. periodic inhibition), now for the
conductance-based RTM neuron instead of the LIF neuron. For each applied
current, the tonic-inhibition branch integrates until either a fixed
point is detected (rate 0) or the 4th spike occurs (rate from the
3rd-to-4th interspike interval), continuing from the previous current's
final state; the periodic branch runs a fixed 2 s window and counts
spikes. The two examples share every gating/step function and
`compute_rtm_f_i_curves`, differing only in the periodic conductance's
peak amplitude ($\bar g$ vs. $2\bar g$). The two sweeps are
`@njit`-compiled (verified against a pre-numba Python version at reduced
scale before trusting it).


In [ ]:
@register_jitable
def rtm_inhib_m_inf(v):
    alpha_m = 0.32 * (v + 54) / (1 - exp(-(v + 54) / 4))
    beta_m = 0.28 * (v + 27) / (exp((v + 27) / 5) - 1)
    return alpha_m / (alpha_m + beta_m)


@register_jitable
def rtm_inhib_alpha_h(v):
    return 0.128 * exp(-(v + 50) / 18)


@register_jitable
def rtm_inhib_beta_h(v):
    return 4. / (1 + exp(-(v + 27) / 5))


@register_jitable
def rtm_inhib_alpha_n(v):
    return 0.032 * (v + 52) / (1 - exp(-(v + 52) / 5))


@register_jitable
def rtm_inhib_beta_n(v):
    return 0.5 * exp(-(v + 57) / 40)


@register_jitable
def rtm_inhib_g_periodic(t, amplitude, alpha, period, norm):
    return amplitude * (exp(alpha * np.cos(np.pi * t / period) ** 2) - 1) / norm


@register_jitable
def rtm_inhib_step(v, m, h, n, i_ext, g_inhib_old, g_inhib_mid,
                    c, g_na, g_k, g_l, v_na, v_k, v_l, v_rev, dt, dt05):
    v_inc = (g_na * m ** 3 * h * (v_na - v) + g_k * n ** 4 * (v_k - v) + g_l * (v_l - v)
             + g_inhib_old * (v_rev - v) + i_ext) / c
    h_inc = rtm_inhib_alpha_h(v) * (1 - h) - rtm_inhib_beta_h(v) * h
    n_inc = rtm_inhib_alpha_n(v) * (1 - n) - rtm_inhib_beta_n(v) * n

    v_tmp = v + dt05 * v_inc
    m_tmp = rtm_inhib_m_inf(v_tmp)
    h_tmp = h + dt05 * h_inc
    n_tmp = n + dt05 * n_inc

    v_inc = (g_na * m_tmp ** 3 * h_tmp * (v_na - v_tmp) + g_k * n_tmp ** 4 * (v_k - v_tmp)
             + g_l * (v_l - v_tmp) + g_inhib_mid * (v_rev - v_tmp) + i_ext) / c
    h_inc = rtm_inhib_alpha_h(v_tmp) * (1 - h_tmp) - rtm_inhib_beta_h(v_tmp) * h_tmp
    n_inc = rtm_inhib_alpha_n(v_tmp) * (1 - n_tmp) - rtm_inhib_beta_n(v_tmp) * n_tmp

    v_new = v + dt * v_inc
    m_new = rtm_inhib_m_inf(v_new)
    h_new = h + dt * h_inc
    n_new = n + dt * n_inc
    return v_new, m_new, h_new, n_new


# njit without cache=True: same reasoning as _periodic_inhibition_f_i_sweep_jit above.
@njit
def _rtm_f_i_curve_tonic_jit(i_ext_values, g_bar, c, g_na, g_k, g_l, v_na, v_k, v_l, v_rev, dt, dt05):
    """For each drive in i_ext_values (continuing from the final state of
    the previous drive), integrate until either a fixed point is
    detected (frequency 0) or the 4th spike occurs (frequency from the
    3rd-to-4th interspike interval)."""
    N_check = round(1000 / dt)
    v, m, h, n = -70., rtm_inhib_m_inf(-70.), 0.7, 0.6
    f_vec = np.zeros(len(i_ext_values))

    for ijk in range(len(i_ext_values)):
        i_ext = i_ext_values[ijk]
        v_hist = [v]
        m_hist = [m]
        h_hist = [h]
        n_hist = [n]
        t_spikes = []
        num_spikes = 0
        k = 0
        f = 0.
        while True:
            k += 1
            v, m, h, n = rtm_inhib_step(v_hist[-1], m_hist[-1], h_hist[-1], n_hist[-1], i_ext,
                                         g_bar, g_bar, c, g_na, g_k, g_l, v_na, v_k, v_l, v_rev, dt, dt05)
            v_hist.append(v)
            m_hist.append(m)
            h_hist.append(h)
            n_hist.append(n)

            if (k - 1) % N_check == 0 and k > 1:
                vv = np.array(v_hist[-N_check - 1:])
                mm = np.array(m_hist[-N_check - 1:])
                hh = np.array(h_hist[-N_check - 1:])
                nn = np.array(n_hist[-N_check - 1:])
                if (vv.max() - vv.min() < 1e-4 * abs(vv.max() + vv.min())
                        and mm.max() - mm.min() < 1e-4 * abs(mm.max() + mm.min())
                        and hh.max() - hh.min() < 1e-4 * abs(hh.max() + hh.min())
                        and nn.max() - nn.min() < 1e-4 * abs(nn.max() + nn.min())):
                    f = 0.
                    break

            if v_hist[-1] < -20 and v_hist[-2] >= -20:
                num_spikes += 1
                t_spikes.append((k * dt * (20 + v_hist[-2]) + (k - 1) * dt * (-20 - v_hist[-1]))
                                 / (v_hist[-2] - v_hist[-1]))
            if num_spikes == 4:
                f = 1000. / (t_spikes[3] - t_spikes[2])
                break

        f_vec[ijk] = f
        v, m, h, n = v_hist[-1], m_hist[-1], h_hist[-1], n_hist[-1]

    return f_vec


@njit
def _rtm_f_i_curve_periodic_jit(i_ext_values, g_amplitude, alpha, period, norm,
                                 c, g_na, g_k, g_l, v_na, v_k, v_l, v_rev, dt, dt05, t_final):
    m_steps = round(t_final / dt)
    t = np.arange(m_steps + 1) * dt
    g_store = rtm_inhib_g_periodic(t, g_amplitude, alpha, period, norm)

    v, m, h, n = -70., rtm_inhib_m_inf(-70.), 0.7, 0.6
    f_vec = np.zeros(len(i_ext_values))

    for ijk in range(len(i_ext_values)):
        i_ext = i_ext_values[ijk]
        num_spikes = 0
        v_old = v
        for k in range(1, m_steps + 1):
            v, m, h, n = rtm_inhib_step(v, m, h, n, i_ext, g_store[k - 1], (g_store[k - 1] + g_store[k]) / 2,
                                         c, g_na, g_k, g_l, v_na, v_k, v_l, v_rev, dt, dt05)
            if v < -20 and v_old >= -20:
                num_spikes += 1
            v_old = v
        f_vec[ijk] = num_spikes / t_final * 1000

    return f_vec


def compute_rtm_f_i_curves(i_ext_values=None, g_amplitude=0.1,
                            alpha=1.0, Period=25.0, g_bar=0.1, N=2000,
                            c=1.0, g_na=100.0, g_k=80.0, g_l=0.1,
                            v_na=50.0, v_k=-100.0, v_l=-67.0, v_rev=-75.0,
                            dt=0.01, t_final=2000.0):
    """F-I curves of the RTM neuron under tonic (g_bar) and under periodic
    inhibition (peak amplitude g_amplitude). Returns
    (i_ext_vec, f_vec_tonic, f_vec_periodic)."""
    if i_ext_values is None:
        i_ext_values = 0. + np.arange(101) / 100 * 3.0
    dt05 = dt / 2
    norm = np.mean(np.exp(alpha * np.cos(np.pi * np.arange(N) / N) ** 2) - 1)

    f_vec_tonic = _rtm_f_i_curve_tonic_jit(i_ext_values, g_bar, c, g_na, g_k, g_l, v_na, v_k, v_l, v_rev, dt, dt05)
    f_vec_periodic = _rtm_f_i_curve_periodic_jit(i_ext_values, g_amplitude, alpha, Period, norm,
                                                  c, g_na, g_k, g_l, v_na, v_k, v_l, v_rev, dt, dt05, t_final)
    return i_ext_values, f_vec_tonic, f_vec_periodic


def compute_rtm_f_i_curves_2(i_ext_values=None):
    return compute_rtm_f_i_curves(i_ext_values=i_ext_values, g_amplitude=0.2)


def plot_rtm_f_i_curves(i_ext_vec, f_vec_tonic, f_vec_periodic):
    fig, ax = plt.subplots(figsize=(7, 5))
    ax.plot(i_ext_vec, f_vec_tonic, '-r', linewidth=2, label='tonic inhibition')
    ax.plot(i_ext_vec, f_vec_periodic, '-b', linewidth=2, label='periodic inhibition')
    ax.axis([i_ext_vec.min(), i_ext_vec.max(), 0, 100])
    ax.set_xlabel(r'$I$ [$\mu$A/cm$^2$]')
    ax.set_ylabel('$f$')
    ax.legend()
    plt.tight_layout()
    return fig

In [ ]:
plot_rtm_f_i_curves(*compute_rtm_f_i_curves());  # RTM_F_I_CURVE_WITH_INHIBITION

In [ ]:
plot_rtm_f_i_curves(*compute_rtm_f_i_curves_2());  # RTM_F_I_CURVE_WITH_INHIBITION_2

In [ ]:
interact(lambda g_amplitude=0.1: plot_rtm_f_i_curves(*compute_rtm_f_i_curves(g_amplitude=g_amplitude)),
         g_amplitude=(0.0, 0.4, 0.02));